# Credit Risk / Churn Prediction

Predicting whether a telecom customer will cancel their service, using the [Telco Customer Churn dataset](https://www.kaggle.com/datasets/blastchar/telco-customer-churn).

**Models compared:** Logistic Regression (baseline) vs. XGBoost (tuned with Optuna), evaluated honestly on a held-out test set, with SHAP explainability on the final model.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, classification_report, confusion_matrix, average_precision_score
)
from xgboost import XGBClassifier
import optuna
import shap

os.makedirs('outputs/charts', exist_ok=True)

## Checkpoint 1 — Load & Inspect the Data

In [ ]:
df = pd.read_csv("telco_churn.csv", index_col="customerID")
df.shape

In [ ]:
df.info()

In [ ]:
df.head()

## Checkpoint 2 — Exploratory Data Analysis

Looking at churn distribution and how key features relate to it, before touching any model.

In [ ]:
# Churn class distribution
sns.countplot(x='Churn', data=df)
plt.title('Churn Distribution')
plt.savefig('outputs/charts/01_churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

df['Churn'].value_counts(normalize=True)

In [ ]:
# Tenure vs churn
sns.boxplot(x='Churn', y='tenure', data=df)
plt.title('Tenure vs Churn')
plt.savefig('outputs/charts/02_tenure_vs_churn.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Monthly charges vs churn
sns.boxplot(x='Churn', y='MonthlyCharges', data=df)
plt.title('Monthly Charges vs Churn')
plt.savefig('outputs/charts/03_monthlycharges_vs_churn.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Contract type vs churn
sns.countplot(x='Contract', hue='Churn', data=df)
plt.title('Contract Type vs Churn')
plt.savefig('outputs/charts/04_contract_vs_churn.png', dpi=150, bbox_inches='tight')
plt.show()

**Findings:** ~75/25 class split (imbalanced). Month-to-month contracts and low tenure are associated with higher churn. No leakage columns identified.

## Checkpoint 3 — Data Cleaning & Preprocessing

In [ ]:
# Fix TotalCharges (loaded as text due to blank entries) and drop unrecoverable rows
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges'])

# Encode target as binary
df['Churn'] = df['Churn'].astype(object).map({'Yes': 1, 'No': 0})

# One-hot encode remaining categorical columns (drop_first avoids redundant/multicollinear columns)
categorical_cols = df.select_dtypes(include=['object', 'string']).columns.tolist()
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

df.dtypes.value_counts()

## Checkpoint 4 — Train/Test Split

80/20 split, stratified to preserve the churn ratio in both sets, with a fixed random state for reproducibility.

In [ ]:
X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train churn ratio:")
print(y_train.value_counts(normalize=True))
print("\nTest churn ratio:")
print(y_test.value_counts(normalize=True))

## Checkpoint 5 — Baseline Model: Logistic Regression

Numeric features are scaled first — logistic regression's gradient-based optimizer converges poorly when features are on very different scales (e.g. `TotalCharges` in the thousands vs. 0/1 dummy columns).

In [ ]:
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled[numeric_cols] = scaler.transform(X_test[numeric_cols])

logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train_scaled, y_train)

In [ ]:
y_pred_proba_lr = logreg.predict_proba(X_test_scaled)[:, 1]
y_pred_lr = logreg.predict(X_test_scaled)

print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba_lr))
print(classification_report(y_test, y_pred_lr))

## Checkpoint 6 — Main Model: XGBoost + Imbalance Handling

XGBoost uses the raw (unscaled) features — tree models split on thresholds and don't need scaling. `scale_pos_weight` corrects for the ~74/26 class imbalance by upweighting the minority (churn) class during training.

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_baseline = XGBClassifier(
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss'
)
xgb_baseline.fit(X_train, y_train)

In [ ]:
y_pred_proba_xgb = xgb_baseline.predict_proba(X_test)[:, 1]
y_pred_xgb = xgb_baseline.predict(X_test)

print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba_xgb))
print(classification_report(y_test, y_pred_xgb))
print(confusion_matrix(y_test, y_pred_xgb))

## Checkpoint 7 — Hyperparameter Tuning (Optuna)

Searching for better XGBoost settings using 5-fold cross-validation on the training set only — the test set stays untouched until Checkpoint 8.

In [ ]:
def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
    }
    model = XGBClassifier(
        **params,
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss'
    )
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')
    return scores.mean()

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50)

print(study.best_params)
print(study.best_value)

In [ ]:
best_params = study.best_params

final_model = XGBClassifier(
    **best_params,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss'
)
final_model.fit(X_train, y_train)

## Checkpoint 8 — Final Evaluation on the Test Set

The one honest, final check — this is the only point where the tuned model touches the held-out test set.

In [ ]:
y_pred_proba_final = final_model.predict_proba(X_test)[:, 1]
y_pred_final = final_model.predict(X_test)

print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba_final))
print("PR-AUC:", average_precision_score(y_test, y_pred_proba_final))
print(classification_report(y_test, y_pred_final))
print(confusion_matrix(y_test, y_pred_final))

**Summary of all three models (test set):**

| Metric | Logistic Regression | XGBoost (untuned) | XGBoost (tuned) |
|---|---|---|---|
| ROC-AUC | 0.851 | 0.834 | 0.835 |
| Precision (churn) | 0.67 | 0.56 | 0.51 |
| Recall (churn) | 0.55 | 0.69 | 0.79 |
| F1 (churn) | 0.60 | 0.62 | 0.62 |

Tuning substantially improved recall on churners (catching more at-risk customers) at the cost of precision and without surpassing logistic regression's ROC-AUC — a genuine tradeoff, not a clean win, discussed further in the project write-up.

## Checkpoint 9 — Explainability (SHAP)

Using `TreeExplainer` for fast, exact SHAP values on the tuned XGBoost model.

In [ ]:
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_test)

In [ ]:
# Global feature importance
shap.summary_plot(shap_values, X_test, show=False)
plt.savefig('outputs/charts/05_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# One individual prediction explained
row_index = 3

shap.force_plot(
    explainer.expected_value,
    shap_values[row_index],
    X_test.iloc[row_index],
    matplotlib=True,
    show=False
)
plt.savefig('outputs/charts/06_shap_force_customer.png', dpi=150, bbox_inches='tight')
plt.show()

**Key drivers found by SHAP:** two-year contracts and longer tenure strongly reduce churn risk; fiber optic internet and electronic check payment method increase it — consistent with and extending the patterns found in the Checkpoint 2 EDA.